# Uplift modeling in marketing. Notebook 3: meta-learners (S/T/X/R)

Continuation of `02_Baseline_Propensity_EN.ipynb`. This notebook does not share the kernel of the previous one; it reloads the training and validation splits and the S3.2 response-targeting baseline (whose score enters the comparative table in S4.3). The sealed test remains hidden and is never accessed in this notebook.

## Contents

- [Setup (resumed from S1 to S3)](#setup)
- [Section 4. Meta-learners (S/T/X/R)](#s4)
    - [4.1 Fitting the four meta-learners](#s4-1)
    - [4.2 Distribution of estimated CATE by learner](#s4-2)
    - [4.3 Preliminary evaluation with uplift metrics in validation](#s4-3)
    - [4.4 Diagnosis of heterogeneity: signal or sampling noise](#s4-4)
        - [4.4.1 Permutation noise floor](#s4-4-1)
        - [4.4.2 GATES-lite: separation of groups with distinct ATE](#s4-4-2)
    - [4.5 Feature iteration and regularization (X/R-learner)](#s4-5)
    - [4.6 Sensitivity to base algorithm](#s4-6)
    - [4.7 Ablation: known versus internally estimated propensity (X/R)](#s4-7)
    - [4.8 Isolating bagging: single tree versus Random Forest at the same depth](#s4-8)
    - [4.9 Stability under repeated resampling](#s4-9)
        - [4.9.1 Stability of `max_depth` (3/4/5)](#s4-9-1)
        - [4.9.2 Paired comparison of the leader against strong competitors](#s4-9-2)
        - [4.9.3 The two leaders against the response-targeting baseline](#s4-9-3)
        - [4.9.4 Causes of Qini AUC variation across samples](#s4-9-4)
    - [4.10 Linear S-learner with explicit T×X interactions (interpretable baseline)](#s4-10)
    - [Synthesis of Section 4](#s4-summary)
- [Glossary (quick reference)](#glossary)

In [ ]:
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.exceptions import ConvergenceWarning

# Path bootstrap: allows `from src...` from the notebooks directory.
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import SEED
from src.i18n import make_lang
from src.viz import apply_plot_style

np.random.seed(SEED)
pd.set_option('display.max_columns', 50)
pd.set_option('display.precision', 4)
warnings.filterwarnings('ignore', category=FutureWarning)
# The internal causalml meta-learner propensity model (S4) uses an
# elastic-net solver that does not converge within the default max_iter on
# small samples. This does not affect the result because it is only a
# nuisance model, but it pollutes the output.
warnings.filterwarnings('ignore', category=ConvergenceWarning)

# Idioma canônico deste notebook é PT — passthrough, sem chamada de rede.
# EN edition: only this line is switched to make_lang('en').
lang = make_lang('en')

apply_plot_style()

In [ ]:
# Stack causal (verifique a instalação antes da primeira run)
# pip install econml causalml scikit-uplift shap mlflow

# Meta-learners e árvores de uplift
from causalml.inference.meta import BaseSRegressor, BaseTRegressor, BaseXRegressor, BaseRRegressor
from causalml.inference.tree import UpliftRandomForestClassifier

# Causal Forest com intervalos de confiança
from econml.dml import CausalForestDML
from econml.metalearners import TLearner, SLearner, XLearner

# Avaliação
from sklift.metrics import uplift_at_k, qini_auc_score, uplift_auc_score

from src.compat import patch_sklearn_matplotlib_support
patch_sklearn_matplotlib_support()
from sklift.viz import plot_qini_curve, plot_uplift_curve

# Interpretabilidade
import shap

In [ ]:
from src.config import ARTIFACTS_DIR, RETRAIN

ARTIFACTS_DIR.mkdir(exist_ok=True, parents=True)

# MLflow — ajuste o URI conforme seu setup Windows
# mlflow.set_tracking_uri('file:///C:/path/to/mlruns')
# mlflow.set_experiment('uplift_hillstrom')

<a id="setup"></a>

## Setup (resumed from S1 to S3)

In the normal flow, `get_train_val` loads the training and validation manifests persisted by `02_Baseline_Propensity_EN` (`train_index.parquet`, `validation_index.parquet`, `dataset_manifest.json`) and validates the SHA-256 fingerprint and `n_rows` of the current dataset against the manifest without rewriting the sealed-test index, which only notebook 2 writes. Recomputing via `train_test_split` acts only as a bootstrap fallback when no split has been persisted yet; once a sealed test exists, that fallback never runs silently (see `src/splits.py`). The S3.2 response-targeting baseline is also refitted because its runtime cost is negligible.

In [ ]:
from src.config import POOLED_TREATMENT_COL, PRIMARY_OUTCOME
from src.data import add_pooled_treatment, load_hillstrom
from src.learners import fit_propensity_baseline, predict_propensity_score
from src.splits import get_train_val

df = load_hillstrom()
df_pooled = add_pooled_treatment(df)
train_df, val_df = get_train_val(df_pooled, persist_test=False)

propensity_model = fit_propensity_baseline(train_df, POOLED_TREATMENT_COL, PRIMARY_OUTCOME)
val_df['propensity_score'] = predict_propensity_score(propensity_model, val_df)

labels = lang({
    'header': 'Tamanho das partições (retomado de S3)',
    'training': 'Treino',
    'validation': 'Validação',
    'rows': 'linhas',
    'treated': 'tratados',
})
print(f"{labels['header']}:")
print(f"  {labels['training']}:   {len(train_df):>6} {labels['rows']} | {labels['treated']}: {int(train_df[POOLED_TREATMENT_COL].sum()):>6}")
print(f"  {labels['validation']}: {len(val_df):>6} {labels['rows']} | {labels['treated']}: {int(val_df[POOLED_TREATMENT_COL].sum()):>6}")

<a id="s4"></a>

# Section 4. Meta-learners (S/T/X/R)

The first objective in this section is estimating CATE with the four classic meta-learners, S-, T-, X-learner (Künzel et al., 2019) and R-learner (Nie & Wager, 2021), using a single base learner, LightGBM (Ke et al., 2017), for all of them. Varying the meta-learner and base algorithm simultaneously would prevent attributing performance differences to the meta-learning strategy or to base-model capacity.

The binary outcome (`visit`) is modeled via Regressor variants (probability regression) across all four learners. Because the `causalml` R-learner exists only as a Regressor, using a Classifier for the other three would create a capacity asymmetry across learners and compromise the comparison. Out-of-sample CATE estimates are then produced on validation for all four estimators, ranked against the response-targeting baseline (S3) under the same uplift metrics, and inspected in distribution. Treatment remains pooled (S3–S6), as established in S3.

> Registered hypothesis before running: the X-learner would hold a structural advantage on this dataset because it was designed specifically for imbalanced treatment arms, matching the 2:1 pooled setup (treated versus control). This hypothesis is tested in Section 4.3; if it is not supported, the negative result is reported as observed (Absolute Rule #6: a negative result is a result).

Note added after 4.3 results: no meta-learner beat the baseline by a clear margin, raising the question of whether the treatment effect is nearly homogeneous in this population or whether the estimators (vanilla LightGBM with raw features) failed to capture real heterogeneity. Three additional subsections investigate that question in order: 4.4 (signal versus noise diagnostics), 4.5 (derived features and regularization for X and R), and 4.6 (sensitivity to the base algorithm). None of them reopens or replaces the decisions of 4.1–4.3; all compare their findings side by side against the original results.

<a id="s4-1"></a>

## 4.1 Fitting the four meta-learners

Categorical variables (`zip_code`, `channel`, `history_segment`) require different handling from the S3 baseline: `causalml` converts `X` to a plain NumPy array before passing it to the base learner, stripping the pandas `category` dtype required by LightGBM's native categorical handling. They are therefore one-hot encoded (fitted on training data only) before entering the four meta-learners.

The workflow follows the project's `RETRAIN` pattern: on first execution, this cell trains and serializes the models to `artifacts/meta_learners.joblib` even when `RETRAIN=False` (bootstrap behavior); subsequent runs load the saved artifact unless `RETRAIN=True`.

In [ ]:
from src.config import META_LEARNERS_PATH
from src.learners import get_meta_learners, predict_meta_learners_uplift

meta_models, meta_encoder = get_meta_learners(
    train_df, POOLED_TREATMENT_COL, PRIMARY_OUTCOME, RETRAIN, META_LEARNERS_PATH,
)
meta_cate = predict_meta_learners_uplift(meta_models, val_df, meta_encoder)
for name, cate in meta_cate.items():
    val_df[f'{name.lower()}_learner_uplift'] = cate

labels = lang({'header': 'CATE estimado — resumo por learner (validação)'})
print(f"{labels['header']}:")
print(val_df[[f'{n.lower()}_learner_uplift' for n in meta_cate]].describe().T)

<a id="s4-2"></a>

## 4.2 Distribution of estimated CATE by learner

Before comparing metrics, the distribution shape of each estimator is examined, as meta-learners have known bias-variance trade-offs: the S-learner tends to shrink effects when treatment enters as one additional feature, whereas the T-learner accumulates higher variance by fitting two independent models.

In [ ]:
from src.viz import plot_uplift_distributions

labels = lang({
    'title': 'Distribuição do CATE estimado por meta-learner',
    'subtitle': 'S-learner encolhe efeitos (desvio menor); T-learner é o mais disperso',
})
fig, ax = plot_uplift_distributions(meta_cate, title=labels['title'], subtitle=labels['subtitle'])
plt.show()

All four distributions have very close means (0.058–0.060), consistent with the pooled ATE of roughly 6pp estimated in S2, as CATE should recover the ATE on average. What separates the estimators is dispersion: the S-learner is visibly the most concentrated (standard deviation 0.036, range -0.15 to +0.42), matching an estimator that shares a single model across arms and shrinks heterogeneity. The T-learner is the most dispersed (standard deviation 0.070, range -0.35 to +0.48), reflecting combined noise from two independent models. The X-learner and R-learner sit between those extremes (standard deviations 0.050 and 0.066, respectively).

<a id="s4-3"></a>

## 4.3 Preliminary evaluation with uplift metrics in validation

The same metrics from S3.3, Qini AUC (Radcliffe & Surry, 2011), uplift AUC (Gutierrez & Gérardy, 2017), and uplift@30%, are applied to the four meta-learners alongside the response-targeting baseline. This is a preliminary validation reading (without bootstrap confidence intervals or opening the sealed test) to guide S5 and S6. The fast T-learner from S3.4 is excluded here as previously stated, having served only as a provisional check.

In [ ]:
from src.evaluation import evaluate_multiple_rankings

all_scores = {
    'Baseline (propensão)': val_df['propensity_score'],
    'S-learner': meta_cate['S'],
    'T-learner': meta_cate['T'],
    'X-learner': meta_cate['X'],
    'R-learner': meta_cate['R'],
}
comparison_table = evaluate_multiple_rankings(
    val_df[PRIMARY_OUTCOME].values, all_scores, val_df[POOLED_TREATMENT_COL].values,
)
print(comparison_table.round(4))

The X-learner hypothesis did not hold: none of the four meta-learners beat the response-targeting baseline by a clear margin in this validation.

For the S-learner, Qini AUC of 0.0415 exceeds the baseline (0.0395) in the third decimal place, too narrow a margin to support model selection on a single holdout; stability is investigated in development data in 4.9. For the X-learner, Qini AUC of 0.0243 fell below the baseline, contradicting the hypothesis recorded in S4 of structural advantage under imbalanced arms; the negative result is reported in full. The T-learner reached Qini AUC of 0.0067, nearly six times below the baseline, and the R-learner reached 0.0196, also below the baseline.

An explanation consistent with the meta-learner literature (Künzel et al., 2019), though unproven here, is that X-learner and R-learner inherit variance from first-stage models: the X-learner builds its pseudo-effects from the same outcome regressions that penalize the standalone T-learner (see `BaseXRegressor.fit` in `causalml`, where `d_c` and `d_t` come from `mu_c` and `mu_t`). Sharing a single model across arms gives the S-learner less variance to inherit, outweighing potential heterogeneity shrinkage.

No protocol adjustment was made to produce an alternative winner (Absolute Rule #6). Candidate stability continues under examination in development data, leaving S6 for confirmatory evaluation of the configuration frozen at the end of S5.

<a id="s4-4"></a>

## 4.4 Diagnosis of heterogeneity: signal or sampling noise

Because no meta-learner beat the baseline with margin in S4.3, two possibilities arise: (a) the treatment effect is nearly homogeneous in this population, so ranking by baseline propensity already captures most of the gain, or (b) the rankings contain structure tied to effect differences that did not emerge clearly in aggregate Qini. The two diagnostics below use the same training and validation data and probe related aspects of that structure, without constituting independent tests or touching the sealed test.

> Interpretation rule recorded before running and later qualified: the initial operational rule treated a real standard deviation above the 95th percentile of the permutation noise floor (4.4.1) as evidence of heterogeneity above sampling noise. After inspecting the null generated by permutation, that reading was qualified: shuffling treatment imposes τ(x)=0 for all x and destroys the ATE as well; exceeding this floor therefore indicates structure beyond the total null, but does not by itself constitute a formal test against H0: τ(x)=ATE (non-zero homogeneous effect). Had no learner exceeded the 95th percentile, the conclusion would remain a homogeneous effect or one not captured by the available features.

<a id="s4-4-1"></a>

### 4.4.1 Permutation noise floor

The `treatment` column is permuted in training (deliberately breaking both heterogeneity and the ATE), only the S-learner is refitted (due to lower computational cost), and `std(CATE)` is measured on the real validation set across 20 repetitions. The resulting distribution represents CATE dispersion caused solely by sampling noise and model overfitting under a complete null with no treatment-outcome association (H0: τ(x)=0 for all x), serving as a reference against the real dispersions measured in 4.2.

This procedure is not a formal heterogeneity test against a non-zero homogeneous effect (H0: τ(x)=ATE), which would require preserving the ATE while perturbing only dependence on X. It is treated explicitly as an exploratory diagnostic restricted to the S-learner for cost; because T, X, and R display larger real dispersion than the S-learner, it is plausible (though not directly verified) that they also exceed their respective noise floors.

In [ ]:
from src.evaluation import permutation_noise_floor

permuted_stds = permutation_noise_floor(
    train_df, val_df, POOLED_TREATMENT_COL, PRIMARY_OUTCOME, meta_encoder,
    learner_name='S', n_reps=20,
)
real_stds = {name: float(cate.std()) for name, cate in meta_cate.items()}

labels = lang({
    'header': 'Chão de ruído por permutação (S-learner, 20 repetições)',
    'real_header': 'Desvios reais por learner',
})
print(f"{labels['header']}:")
print(f"  min={permuted_stds.min():.4f}  p50={pd.Series(permuted_stds).median():.4f}  "
    f"p95={pd.Series(permuted_stds).quantile(0.95):.4f}  max={permuted_stds.max():.4f}")
print(f"\n{labels['real_header']}: {({k: round(v, 4) for k, v in real_stds.items()})}")

In [ ]:
from src.viz import plot_permutation_noise_floor

labels = lang({
    'title': 'Chão de ruído por permutação vs. dispersão real do CATE',
    'subtitle': 'O desvio real do S-learner (0.036) fica acima do chão de ruído (máx. 0.029)',
})
fig, ax = plot_permutation_noise_floor(permuted_stds, real_stds, title=labels['title'], subtitle=labels['subtitle'])
plt.show()

The permutation noise floor for the S-learner (20 repetitions) ranged from 0.0166 to 0.0290 (p95 = 0.0273, mean 0.0228). The S-learner's real standard deviation (0.0359, measured in 4.2) exceeded the maximum observed in the noise floor and stood well above the 95th percentile.

Observed CATE dispersion exceeds that produced under a null that completely destroys the treatment-outcome association, suggesting structure beyond pure noise. However, this diagnostic does not constitute a formal test against a homogeneous non-zero treatment effect: because permutation zeros the ATE along with heterogeneity, exceeding this floor is consistent with both genuine heterogeneity and asymmetric estimation noise under ATE ≠ 0 versus ATE = 0, a distinction not resolved here. The other three learners show larger real dispersions (T = 0.0705, X = 0.0500, R = 0.0661), an indication not tested individually due to cost.

This reading qualifies S4.3 without settling the question: the modest Qini AUC neither confirms a homogeneous effect nor confirms τ(x) varying with X. A permutation test preserving the ATE remains future work, motivating feature engineering and regularization in 4.5 before discarding the meta-learners.

<a id="s4-4-2"></a>

### 4.4.2 GATES-lite: separation of groups with distinct ATE

Following a simplified version of Grouped Average Treatment Effects (Chernozhukov et al., 2018b), validation rows are ranked by each learner's estimated CATE, split into quintiles, and evaluated by ATE (Imbens & Rubin, 2015) within each quintile (via `effects.ate_binary`, the same function used in S2.5) with 95% confidence intervals. If the top predicted-CATE quintile achieves a higher real ATE separable from the bottom quintile (non-overlapping CIs), the ranking carries useful heterogeneity signal even without a clear advantage in aggregate Qini.

In [ ]:
from src.evaluation import gates_by_cate_quintile

gates_tables = {
    name: gates_by_cate_quintile(val_df, f'{name.lower()}_learner_uplift', PRIMARY_OUTCOME, POOLED_TREATMENT_COL)
    for name in meta_cate
}
for name, gates_df in gates_tables.items():
    print(f"\n{name}-learner:")
    print(gates_df.round(4))

In [ ]:
from src.viz import plot_gates_bars

labels = lang({
    'title': 'GATES-lite — ATE por quintil de CATE estimado (S-learner)',
    'subtitle': 'ATE cresce do quintil 0 ao 4 (0.049 -> 0.105), mas os ICs quase se tocam',
})
fig, ax = plot_gates_bars(gates_tables['S'], title=labels['title'], subtitle=labels['subtitle'])
plt.show()

Across all four learners, the highest predicted-CATE quintile (group 4) exhibited the highest ATE among the five groups, with an approximately increasing trend (in the S-learner, from 0.049 in group 0 to 0.105 in group 4). The pattern suggests signal in the rankings, though all four models are evaluated on the same individuals and produce correlated rankings rather than four independent confirmations.

Under the strict non-overlapping confidence-interval criterion, evidence is borderline: in the S-learner, group 0's CI is [0.021, 0.078] and group 4's is [0.077, 0.133], overlapping by 0.0008. In T, X, and R, overlap between extremes is wider. With roughly 2,560 rows per quintile and a modest baseline visit rate, statistical power to separate quintiles is constrained by the data.

Both diagnostics point in the same direction without forming a confirmatory heterogeneity test: 4.4.1 shows dispersion above the total null, and 4.4.2 shows rising ATE from quintile 0 to 4 with borderline separation under individual CIs. Whether the structure reflects genuine heterogeneity or a homogeneous non-zero ATE remains open in this section, motivating feature and regularization checks (4.5) and base-algorithm sensitivity (4.6).

In [ ]:
from scipy.stats import spearmanr

from src.evaluation import gates_delta_bootstrap

gates_delta_results = {}
for name in meta_cate:
    col = f'{name.lower()}_learner_uplift'
    result = gates_delta_bootstrap(val_df, col, PRIMARY_OUTCOME, POOLED_TREATMENT_COL, n_groups=5, n_boot=2000)
    rho, rho_p = spearmanr(gates_tables[name]['group'], gates_tables[name]['ate'])
    result['spearman_rho'] = rho
    result['spearman_p'] = rho_p
    gates_delta_results[name] = result

labels = lang({'header': 'Δ_GATES = ATE(quintil topo) − ATE(quintil base), IC 95% via bootstrap (n_boot=2000)'})
print(f"{labels['header']}:\n")
for name, r in gates_delta_results.items():
    print(f"{name}-learner: Δ_GATES={r['delta_gates']:.4f}  IC95%=[{r['ci_low']:.4f}, {r['ci_high']:.4f}]  "
          f"p={r['p_value']:.4f}  |  tendência monótona: rho={r['spearman_rho']:.3f} p={r['spearman_p']:.4f}")

The direct contrast `gates_delta_bootstrap` estimates the distribution of Δ_GATES = ATE(quintile 4) − ATE(quintile 0) via bootstrap (Efron & Tibshirani, 1993) within each quintile, preserving random assignment within each fixed stratum:

| Learner | Δ_GATES | 95% CI | p-value | Monotonic trend (Spearman, quintile × ATE) |
|---|---|---|---|---|
| S | 0.0557 | [0.0153, 0.0950] | 0.0080 | ρ = 0.600, p = 0.2848 |
| T | 0.0170 | [-0.0247, 0.0597] | 0.4110 | ρ = 0.600, p = 0.2848 |
| X | 0.0434 | [0.0052, 0.0835] | 0.0250 | ρ = 0.400, p = 0.5046 |
| R | 0.0263 | [-0.0143, 0.0691] | 0.2170 | ρ = 0.900, p = 0.0374 |

For S and X, the CI excludes zero and indicates statistically significant separation between the top and bottom quintiles, a higher-powered result than the informal CI overlap check above. For T and R, the CI includes zero, providing no direct evidence of separation between extremes. Spearman correlation across only 5 points has low statistical power; for the R-learner, ρ = 0.900 (p = 0.0374) occurs in the very learner whose Δ_GATES is not statistically significant, showing that monotonicity and extreme separation measure different aspects.

These four results are not independent pieces of evidence, as all learners evaluate the same ~12,800 individuals in `val_df` using the same eight covariates; two p-values below 0.05 reflect a single underlying signal captured with unequal strength, with direct evidence concentrated in S and X. Furthermore, `gates_delta_bootstrap` resamples rows within fixed quintiles, measuring uncertainty conditional on the observed groups without incorporating estimator refitting and quintile reassignment variance.

<a id="s4-5"></a>

## 4.5 Feature iteration and regularization (X/R-learner)

Because X-learner and R-learner fell below the baseline in 4.3 (0.0243 and 0.0196 versus 0.0395), two interventions documented in the literature are tested before discarding them: derived features and regularization to contain first-stage noise amplification (Künzel et al., 2019) and (Nie & Wager, 2021). S-learner and T-learner remain unchanged as the 4.1–4.3 reference.

The four derived features defined in the isolated module `src/features.py` (without altering global `FEATURE_COLS` or `CAT_VARS`) are: `history_per_recency = history / (recency + 1)`, a ratio not directly reconstructed by orthogonal splits that separates high-spend recent buyers from lapsed ones; `newbie_x_channel` (6 levels), capturing channel differences between new and established customers; `mens_and_womens` (binary), flagging buyers across both merchandise categories; and `zip_code_x_channel` (9 levels), combining geography and digital channel.

Regularization in `_regularized_base_learner` reduces tree complexity while preserving `n_estimators` and `learning_rate`: `num_leaves=15` (default 31), `max_depth=5` (default unlimited), `min_child_samples=50` (default 20), and `reg_alpha=reg_lambda=1.0` (default 0).

> Preregistered structure without an adoption threshold: exactly three configurations are evaluated without grid search: (A) derived features with default hyperparameters; (B) original features with regularization; and (C) both combined. All three enter the comparative table alongside 4.3 results without overwriting prior entries, advancing the highest validation Qini AUC as a candidate for S5/S6 while leaving formal confirmation to S6 bootstrap.

Computational cost: fitting all three configurations for X and R (6 fits across 38,400 training rows) takes a few minutes, measured directly in the cell below without caching or reuse in later sections.

In [ ]:
from src.features import EXTENDED_BIN_VARS, EXTENDED_CAT_VARS, EXTENDED_CONT_VARS, add_engineered_features
from src.learners import _default_base_learner, build_meta_learner_encoder, fit_meta_learners_regularized

train_ext = add_engineered_features(train_df)
val_ext = add_engineered_features(val_df)
encoder_ext = build_meta_learner_encoder(train_ext, cat_vars=EXTENDED_CAT_VARS)

s45_timing = {}

# Config A: features estendidas + hiperparâmetros vanilla (isola o efeito das features)
t0 = time.time()
models_a = fit_meta_learners_regularized(
    train_ext, POOLED_TREATMENT_COL, PRIMARY_OUTCOME, encoder_ext,
    cont_vars=EXTENDED_CONT_VARS, bin_vars=EXTENDED_BIN_VARS, cat_vars=EXTENDED_CAT_VARS,
    base_learner_factory=lambda: _default_base_learner(SEED),
)
s45_timing['A'] = time.time() - t0
cate_a = predict_meta_learners_uplift(
    models_a, val_ext, encoder_ext,
    cont_vars=EXTENDED_CONT_VARS, bin_vars=EXTENDED_BIN_VARS, cat_vars=EXTENDED_CAT_VARS,
)

# Config B: features originais + hiperparâmetros regularizados (isola o efeito da regularização)
t0 = time.time()
models_b = fit_meta_learners_regularized(train_df, POOLED_TREATMENT_COL, PRIMARY_OUTCOME, meta_encoder)
s45_timing['B'] = time.time() - t0
cate_b = predict_meta_learners_uplift(models_b, val_df, meta_encoder)

# Config C: features estendidas + hiperparâmetros regularizados (combinação)
t0 = time.time()
models_c = fit_meta_learners_regularized(
    train_ext, POOLED_TREATMENT_COL, PRIMARY_OUTCOME, encoder_ext,
    cont_vars=EXTENDED_CONT_VARS, bin_vars=EXTENDED_BIN_VARS, cat_vars=EXTENDED_CAT_VARS,
)
s45_timing['C'] = time.time() - t0
cate_c = predict_meta_learners_uplift(
    models_c, val_ext, encoder_ext,
    cont_vars=EXTENDED_CONT_VARS, bin_vars=EXTENDED_BIN_VARS, cat_vars=EXTENDED_CAT_VARS,
)

s45_scores = {
    'Baseline (propensão)': val_df['propensity_score'],
    'S-learner (original)': meta_cate['S'],
    'T-learner (original)': meta_cate['T'],
    'X-learner (original)': meta_cate['X'],
    'R-learner (original)': meta_cate['R'],
    'X-learner (A: features)': cate_a['X'],
    'R-learner (A: features)': cate_a['R'],
    'X-learner (B: regularizado)': cate_b['X'],
    'R-learner (B: regularizado)': cate_b['R'],
    'X-learner (C: features+reg)': cate_c['X'],
    'R-learner (C: features+reg)': cate_c['R'],
}
s45_table = evaluate_multiple_rankings(val_df[PRIMARY_OUTCOME].values, s45_scores, val_df[POOLED_TREATMENT_COL].values)
print(s45_table.round(4))
labels = lang({'header': 'Tempo de fit por configuração (X+R cada)'})
print()
print(f"{labels['header']}: {({k: round(v, 1) for k, v in s45_timing.items()})}")

None of the six combinations (X/R × A/B/C) outperformed the original S-learner (0.0415) or the baseline (0.0395), leaving the leading candidate from 4.3 unchanged. However, the per-intervention effects were specific and sizable.

Isolated regularization (B) was the best intervention for the X-learner, raising Qini AUC from 0.0243 to 0.0347, consistent with limiting first-stage noise propagation through shallower trees. For the R-learner, isolated derived features (A) delivered the largest gain, from 0.0196 to 0.0352. Conversely, each intervention helped the opposite learner little or hurt it: derived features dropped the X-learner from 0.0243 to 0.0013, likely because one-hot expansion (6 and 9 levels) enlarged overfitting of unregularized pseudo-effects, whereas regularization (B) gave the R-learner only a marginal lift (0.0196 to 0.0242). Combining both in (C) worsened both learners relative to their best isolated configurations, pushing the R-learner to 0.0162 (below the original 0.0196) without a proven explanation in this round.

All six configurations appear in the table without retrospective selection (Absolute Rule #6). Each meta-learner responded to a different remedy, and combining both interventions naively failed to sum their individual gains.

<a id="s4-6"></a>

## 4.6 Sensitivity to base algorithm

Up to this point, only LightGBM (tree gradient boosting) has been tested as the base algorithm. This section examines whether the S4 results are specific to LightGBM or persist when switching to three other algorithm families, each represented once:

| Family | Learner | Rationale |
|---|---|---|
| Linear / regularized | `ElasticNet` | Tests whether CATE is approximated by an additive linear surface (Zou & Hastie, 2005) |
| Single tree | `DecisionTreeRegressor(max_depth=4)` | Regression tree (Breiman et al., 1984) with depth constrained to prevent variance explosion |
| Tree ensemble (bagging) | `RandomForestRegressor` | Variance reduction by decorrelation with random forests (Breiman, 2001) instead of sequential boosting |

A single axis is varied without crossing 4.5: the original 8 covariates, default hyperparameters for each family (with only `random_state` fixed), and all four meta-learners (S, T, X, and R).

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import ElasticNet
from sklearn.tree import DecisionTreeRegressor
from src.learners import fit_meta_learners

families = {
    'Linear (ElasticNet)': lambda: ElasticNet(random_state=SEED),
    'Árvore única (max_depth=4)': lambda: DecisionTreeRegressor(max_depth=4, random_state=SEED),
    'Random Forest': lambda: RandomForestRegressor(random_state=SEED),
}

family_cate = {}
family_timing = {}
for family_name, factory in families.items():
    t0 = time.time()
    models = fit_meta_learners(
        train_df, POOLED_TREATMENT_COL, PRIMARY_OUTCOME, meta_encoder, base_learner_factory=factory,
    )
    family_timing[family_name] = time.time() - t0
    family_cate[family_name] = predict_meta_learners_uplift(models, val_df, meta_encoder)

s46_scores = {'Baseline (propensão)': val_df['propensity_score']}
for name in ['S', 'T', 'X', 'R']:
    s46_scores[f'{name}-learner (LightGBM)'] = meta_cate[name]
for family_name, cate in family_cate.items():
    for name in ['S', 'T', 'X', 'R']:
        s46_scores[f'{name}-learner ({family_name})'] = cate[name]

s46_table = evaluate_multiple_rankings(val_df[PRIMARY_OUTCOME].values, s46_scores, val_df[POOLED_TREATMENT_COL].values)
print(s46_table.round(4))
labels = lang({'header': 'Tempo de fit por família (4 meta-learners cada)'})
print(f"\n{labels['header']}: {({k: round(v, 1) for k, v in family_timing.items()})}")

The base algorithm had a material impact in this validation, with distinct behavior across the three families.

The S-learner with `ElasticNet` yielded Qini AUC = 0.0000 exactly by mathematical construction: in `causalml` (`BaseSRegressor.fit`/`predict`), treatment enters as an added column in `X` (`np.hstack((w, X))`) with no covariate interactions, producing `CATE = μ(x,1) − μ(x,0) = β_treatment`, a constant across all individuals. Without score variation, Qini AUC is zero by definition, unlike T, X, and R, which fit separate arm models.

A single shallow tree (`max_depth=4`) achieved the best results across Section 4: the X-learner reached Qini AUC = 0.0627 (+58.7% or 1.59× over the 0.0395 baseline), surpassing the prior best (S-learner with LightGBM, 0.0415), with clear gains for T and S as well. This aligns with structure beyond pure noise (subject to the 4.4.1 caveats) and indicates that simpler, constrained models capture the signal better on this single split than vanilla LightGBM.

`RandomForestRegressor` performed poorly, with negative Qini AUC for S (-0.0106) and T (-0.0096). An untested explanation is that unconstrained trees overfit the training data and averaging overfitted trees fails to cancel shared overfitting bias.

Without protocol changes (Absolute Rule #6), the highest validation Qini AUC across Section 4 becomes X-learner with a shallow single tree (0.0627). This remains a single-split validation candidate without confidence intervals or sealed-test confirmation; stability under resampling is tested in 4.9 before any selection.

<a id="s4-7"></a>

## 4.7 Ablation: known versus internally estimated propensity (X/R)

In the Hillstrom RCT (Hillstrom, 2008), by design, the propensity score (Rosenbaum & Rubin, 1983) $P(T{=}1 \mid X) = P(T{=}1) = 2/3$ (pooled) does not depend on $X$. Nevertheless, `fit_single_meta_learner` calls `causalml` without passing `p`, causing X- and R-learners to estimate propensity internally via `ElasticNetPropensityModel`, a potential source of unnecessary nuisance noise.

Inspection of the `causalml` 0.15.5 source code reveals mechanical differences in how `p` is handled: in the X-learner, passing `p` to `fit` leaves `mu_c`, `mu_t`, `tau_c`, and `tau_t` unchanged and only prevents setting `self.propensity_model`; in `predict`, `p` is required to compute `te = p·tau_c_hat + (1-p)·tau_t_hat`, otherwise `TypeError` is raised (because `self.propensity_model` is `None`). Hence, the ablation passes `p` to both `fit` and `predict`. In the R-learner, `p` in `fit` enters directly into the R loss (`(y - ŷ)/(w - p)`, weight `(w-p)²`), changing what `models_tau` learns, while `p` in `predict` is accepted in the signature and unused in the method body.

Hypothesis: fixing `p = 2/3` (theoretical design value) instead of estimating it internally reduces nuisance noise and improves X and/or R. Only the propensity input (internal versus fixed) varies, tested across two base learners: the shallow tree (`max_depth=4`) and the regularized LightGBM from 4.5.

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor

from src.learners import encode_meta_learner_features, fit_single_meta_learner, predict_single_meta_learner

tree_factory = lambda: DecisionTreeRegressor(max_depth=4, random_state=SEED)
p_train_fixed = np.full(len(train_df), 2 / 3)
p_val_fixed = np.full(len(val_df), 2 / 3)

X_train_full = encode_meta_learner_features(train_df, meta_encoder)
X_val_full = encode_meta_learner_features(val_df, meta_encoder)
treatment_arr = train_df[POOLED_TREATMENT_COL].to_numpy()
y_arr = train_df[PRIMARY_OUTCOME].to_numpy(dtype=float)

s47_scores = {'Baseline (propensão)': val_df['propensity_score']}
internal_propensity_stats = {}

for name in ['X', 'R']:
    model_a = fit_single_meta_learner(name, X_train_full, treatment_arr, y_arr, seed=SEED, base_learner_factory=tree_factory)
    s47_scores[f'{name}-learner árvore (A: interna)'] = predict_single_meta_learner(name, model_a, X_val_full)

    if hasattr(model_a, 'propensity') and model_a.propensity:
        group = list(model_a.propensity.keys())[0]
        p_internal = model_a.propensity[group]
        internal_propensity_stats[name] = {
            'mean': float(np.mean(p_internal)), 'std': float(np.std(p_internal)),
            'min': float(np.min(p_internal)), 'max': float(np.max(p_internal)),
        }

    model_b = fit_single_meta_learner(
        name, X_train_full, treatment_arr, y_arr, seed=SEED, base_learner_factory=tree_factory, p=p_train_fixed,
    )
    s47_scores[f'{name}-learner árvore (B: fixa 2/3)'] = predict_single_meta_learner(name, model_b, X_val_full, p=p_val_fixed)

labels = lang({'header': 'Dispersão da propensão interna estimada (deveria ser ~constante 2/3 num RCT)'})
print(f"{labels['header']}:")
for name, stats in internal_propensity_stats.items():
    print(f"  {name}-learner: {({k: round(v, 4) for k, v in stats.items()})}")

s47_table = evaluate_multiple_rankings(val_df[PRIMARY_OUTCOME].values, s47_scores, val_df[POOLED_TREATMENT_COL].values)
print()
print(s47_table.round(4))

In [ ]:
# Repeats the ablation with the regularized LightGBM from 4.5 to check whether
# the tree result generalizes to another base learner.
from src.learners import _regularized_base_learner

lgbm_reg_factory = lambda: _regularized_base_learner(SEED)

s47_lgbm_scores = {'Baseline (propensão)': val_df['propensity_score']}
for name in ['X', 'R']:
    model_a = fit_single_meta_learner(name, X_train_full, treatment_arr, y_arr, seed=SEED, base_learner_factory=lgbm_reg_factory)
    s47_lgbm_scores[f'{name}-learner LightGBM-reg (A: interna)'] = predict_single_meta_learner(name, model_a, X_val_full)

    model_b = fit_single_meta_learner(
        name, X_train_full, treatment_arr, y_arr, seed=SEED, base_learner_factory=lgbm_reg_factory, p=p_train_fixed,
    )
    s47_lgbm_scores[f'{name}-learner LightGBM-reg (B: fixa 2/3)'] = predict_single_meta_learner(name, model_b, X_val_full, p=p_val_fixed)

s47_lgbm_table = evaluate_multiple_rankings(val_df[PRIMARY_OUTCOME].values, s47_lgbm_scores, val_df[POOLED_TREATMENT_COL].values)
print(s47_lgbm_table.round(4))

Results were mostly negative and are reported as observed. Internally estimated propensity from `ElasticNetPropensityModel` had mean 0.6671 (matching the training treatment rate) and standard deviation 0.0109 (range 0.444 to 0.900), showing modest individual variation around the theoretical constant.

With the shallow tree (`max_depth=4`), the X-learner barely changed (0.0627 → 0.0615, difference -0.0012) and the R-learner rose slightly (0.0453 → 0.0459, +0.0006). With regularized LightGBM, the X-learner remained identical (0.0347 → 0.0347) and the R-learner improved from 0.0242 to 0.0291 (+0.0049). The mechanical pattern is consistent across base learners: in the X-learner, `p` only reweights final predictions in `predict`; in the R-learner, `p` enters the training loss directly, producing a small, consistent gain insufficient to alter section leadership (0.0627).

This finding does not invalidate the nuisance-noise hypothesis from the literature (Künzel et al., 2019) and (Nie & Wager, 2021): it only isolates that propensity-model noise is too small to explain the weaker X/R performance, leaving outcome nuisance models (`mu_c`/`mu_t` in `d_c`/`d_t`) open as a candidate explanation. Limitations include single-split validation, a single random seed, and no formal significance testing or compensatory tuning (Absolute Rule #6).

<a id="s4-8"></a>

## 4.8 Isolating bagging: single tree versus Random Forest at the same depth

In 4.6, Random Forest with default hyperparameters (`max_depth=None`) produced negative Qini for S and T, whereas the single tree used `max_depth=4`. To separate bagging from unrestricted depth, both models are set to `max_depth=4` with identical `random_state`, leaving all other `RandomForestRegressor` parameters at `scikit-learn` defaults (printed below). The tested hypothesis expects bagging on already shallow trees to at least match the single tree by reducing variance.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor

rf_defaults = RandomForestRegressor().get_params()
labels = lang({'header': 'Hiperparâmetros do RandomForestRegressor mantidos no default'})
print(f"{labels['header']}:")
for k in ['n_estimators', 'max_depth', 'max_features', 'bootstrap', 'min_samples_leaf', 'min_samples_split']:
    print(f"  {k} = {rf_defaults[k]}")

bagging_families = {
    'Árvore única (depth=4)': lambda: DecisionTreeRegressor(max_depth=4, random_state=SEED),
    'Random Forest (max_depth=4)': lambda: RandomForestRegressor(max_depth=4, random_state=SEED),
}

bagging_cate = {}
for family_name, factory in bagging_families.items():
    models = fit_meta_learners(train_df, POOLED_TREATMENT_COL, PRIMARY_OUTCOME, meta_encoder, base_learner_factory=factory)
    bagging_cate[family_name] = predict_meta_learners_uplift(models, val_df, meta_encoder)

s48_scores = {'Baseline (propensão)': val_df['propensity_score']}
for family_name, cate in bagging_cate.items():
    for name in ['S', 'T', 'X', 'R']:
        s48_scores[f'{name}-learner ({family_name})'] = cate[name]

s48_table = evaluate_multiple_rankings(val_df[PRIMARY_OUTCOME].values, s48_scores, val_df[POOLED_TREATMENT_COL].values)
print()
print(s48_table.round(4))

With depth equalized at `max_depth=4`, results change substantially relative to 4.6, yet bagging provides no uniform gain and worsens the leader (X):

| Learner | Single tree (depth=4) | Random Forest (depth=4) | Difference |
|---|---|---|---|
| S | 0.0515 | 0.0521 | +0.0006 (practical tie) |
| T | 0.0615 | 0.0524 | -0.0091 |
| X | 0.0627 | 0.0538 | -0.0089 |
| R | 0.0453 | 0.0492 | +0.0039 |

First, the collapse of Random Forest in 4.6 came from unrestricted depth rather than bagging: at `max_depth=4`, all four learners become positive. Second, bagging failed to beat the single tree for X (-0.0089) and T (-0.0091). One untested explanation is that averaging across bootstrap resamples smooths threshold splits located more sharply by a single tree on the full training set, helping squared error without improving ranking metrics such as Qini AUC. This comparison ran only on the fixed holdout without the repeated resampling used in 4.9.

As a methodological limitation, `max_features=1.0` is the default in the installed `scikit-learn` `RandomForestRegressor`: every tree evaluates all features at each split, so decorrelation comes solely from row bootstrapping rather than feature subsampling as in Breiman's classic formulation (Breiman, 2001). Testing `max_features='sqrt'` stays out of scope to avoid opening a new tuning axis, and the X-learner drop is reported without protocol changes (Absolute Rule #6).

<a id="s4-9"></a>

## 4.9 Stability under repeated resampling

Comparisons from 4.1 to 4.8 relied on a single fixed train-to-validation split, making every Qini AUC dependent on one sample. To test whether the current leader (X-learner with shallow tree, Qini AUC = 0.0627) holds across other partitions, a repeated stratified holdout protocol is executed exclusively inside `train_df` (keeping `val_df` and the sealed test untouched). Fifteen 75%/25% partitions stratified by (treatment × outcome) are drawn with seeds `1000+rep`, fitting each candidate on 75% and evaluating on 25%. Unlike k-fold/OOF, observations may appear 0, 1, or multiple times across evaluation sets; reusing the same 15 splits and seeds enables paired differences per split (Δ_r = metric(candidate,r) − metric(reference,r)).

Three comparisons proceed in order: (4.9.1) stability of `max_depth` in `{3, 4, 5}`; (4.9.2) paired comparison of the leader against strong competitors from 4.6–4.8; and (4.9.3) the two top candidates from 4.9.2 against the response-targeting baseline on the same splits. Section 4.10 returns to the fixed holdout solely for the predefined interpretable linear baseline.

<a id="s4-9-1"></a>

### 4.9.1 Stability of `max_depth` (3/4/5)

If `max_depth=4` were consistently superior, its average Qini AUC under resampling should exceed `max_depth=3` and `max_depth=5` with low distributional overlap. Only tree depth in the X-learner (`3`, `4`, and `5`) varies across the same data, resampling protocol, and `random_state`.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

from src.evaluation import repeated_holdout_summary, repeated_stratified_holdout

depth_candidates = {
    f'depth={d}': ('meta', 'X', (lambda dd=d: DecisionTreeRegressor(max_depth=dd, random_state=SEED)), False)
    for d in [3, 4, 5]
}
depth_results = repeated_stratified_holdout(train_df, POOLED_TREATMENT_COL, PRIMARY_OUTCOME, depth_candidates, n_reps=15)
depth_summary = repeated_holdout_summary(depth_results)

labels = lang({'header': 'Estabilidade de max_depth — X-learner, 15 repeated stratified holdouts (só em train_df)'})
print(f"{labels['header']}:")
print(depth_summary.round(4))

There is no evidence of a stable advantage for any depth over the others in this resampling; `max_depth=4` is retained.

| Depth | Mean | Median | Standard deviation | Win rate |
|---|---|---|---|---|
| 3 | 0.0254 | 0.0296 | 0.0124 | 26.7% |
| 4 | 0.0250 | 0.0259 | 0.0165 | 40.0% |
| 5 | 0.0269 | 0.0275 | 0.0099 | 33.3% |

Means differ by only 0.0019 between the highest (`depth=5`, 0.0269) and lowest (`depth=4`, 0.0250), far smaller than each standard deviation (0.0099 to 0.0165). Although `depth=5` has the smallest standard deviation and `depth=4` the highest split win rate (40.0%), no depth dominates across criteria. `max_depth=4` is maintained for continuity with 4.6–4.8, as empirical evidence does not justify changing the protocol (Absolute Rule #6). Secondary metrics (Uplift AUC and Uplift@30%) show the same practical tie.

<a id="s4-9-2"></a>

### 4.9.2 Paired comparison of the leader against strong competitors

If X-learner with a shallow tree is genuinely the strongest architecture in S4, it should beat the strongest competitors across most of the 15 resampled splits with positive paired differences. The same splits evaluate: the leader (`X+Tree(depth=4)`); `T+Tree(depth=4)` (which reached Qini AUC = 0.0615 on the fixed holdout); `S+LightGBM(vanilla)` (best result from 4.1–4.3); and the two regularized variants from 4.5 and 4.7 (`X` and `R` with regularized LightGBM, `R` with `p = 2/3`).

In [ ]:
from src.evaluation import paired_deltas
from src.learners import _regularized_base_learner

tree_factory = lambda: DecisionTreeRegressor(max_depth=4, random_state=SEED)
lgbm_reg_factory = lambda: _regularized_base_learner(SEED)

paired_candidates = {
    'X+Tree(depth=4)': ('meta', 'X', tree_factory, False),
    'T+Tree(depth=4)': ('meta', 'T', tree_factory, False),
    'S+LightGBM(vanilla)': ('meta', 'S', None, False),
    'X+LightGBM(regularizado)': ('meta', 'X', lgbm_reg_factory, False),
    'R+LightGBM(regularizado,p=2/3)': ('meta', 'R', lgbm_reg_factory, True),
}
paired_results = repeated_stratified_holdout(train_df, POOLED_TREATMENT_COL, PRIMARY_OUTCOME, paired_candidates, n_reps=15)
paired_summary = repeated_holdout_summary(paired_results)

labels = lang({'header': 'Comparação pareada — 15 repeated stratified holdouts (só em train_df)'})
print(f"{labels['header']}:")
print(paired_summary.round(4))

deltas_vs_leader = paired_deltas(paired_results, baseline_candidate='X+Tree(depth=4)')
deltas_vs_leader[['delta_mean', 'delta_median']] *= -1
deltas_vs_leader['prop_delta_positive'] = 1 - deltas_vs_leader['prop_delta_positive']
print('\nΔ = Qini(X+Tree(depth=4)) − Qini(candidato), por split:')
print(deltas_vs_leader.round(4))

The leader holds an advantage over most competitors, but not over the S-learner with vanilla LightGBM.

| Candidate | Mean | Median | Win rate |
|---|---|---|---|
| X+Tree(depth=4) | 0.0250 | 0.0259 | 46.7% |
| S+LightGBM(vanilla) | 0.0232 | 0.0228 | 26.7% |
| X+LightGBM(regularized) | 0.0165 | 0.0163 | 6.7% |
| R+LightGBM(regularized,p=2/3) | 0.0129 | 0.0158 | 20.0% |
| T+Tree(depth=4) | 0.0039 | 0.0013 | 0.0% |

Paired differences (`X+Tree` minus each competitor, per split):

| vs. | Mean Δ | Median Δ | P(Δ>0) |
|---|---|---|---|
| T+Tree(depth=4) | +0.0211 | +0.0197 | 93% |
| R+LightGBM(regularized,p=2/3) | +0.0121 | +0.0069 | 67% |
| X+LightGBM(regularized) | +0.0085 | +0.0044 | 67% |
| S+LightGBM(vanilla) | +0.0018 | +0.0013 | 53% |

Against `T+Tree(depth=4)` (which fell from 0.0615 on the fixed holdout to mean 0.0039 under resampling) and both regularized LightGBM variants, `X+Tree(depth=4)` wins in 67% to 93% of splits with positive mean Δ. Against `S+LightGBM(vanilla)`, however, the leader wins in only 53% of splits with mean Δ of 0.0018, small relative to the standard deviations (0.0165 and 0.0115); paired Wilcoxon and t-tests give p = 0.89 and p = 0.71 (secondary diagnostics subject to overlapping splits).

X-learner with a shallow tree and S-learner with vanilla LightGBM therefore form a top tier without stable separation between them. The 0.0627 Qini observed on the single holdout did not reproduce under resampling, though the repeated protocol uses smaller training and evaluation samples; both candidates remain equally valid moving into S5.

<a id="s4-9-3"></a>

### 4.9.3 The two leaders against the response-targeting baseline

This subsection checks whether the two leading candidates (`X+Tree(depth=4)` and `S+LightGBM(vanilla)`) show a consistent advantage over the simple response-targeting baseline (`fit_propensity_baseline`/`predict_propensity_score`) when evaluated on the same 15 splits and seeds.

In [ ]:
baseline_candidates = {
    'X+Tree(depth=4)': ('meta', 'X', tree_factory, False),
    'S+LightGBM(vanilla)': ('meta', 'S', None, False),
    'Baseline (propensão)': ('propensity', None, None, False),
}
baseline_results = repeated_stratified_holdout(train_df, POOLED_TREATMENT_COL, PRIMARY_OUTCOME, baseline_candidates, n_reps=15)
baseline_summary = repeated_holdout_summary(baseline_results)

labels = lang({'header': 'Líderes vs. baseline de propensão — 15 repeated stratified holdouts (só em train_df)'})
print(f"{labels['header']}:")
print(baseline_summary.round(4))

deltas_vs_baseline = paired_deltas(baseline_results, baseline_candidate='Baseline (propensão)')
print('\nΔ = Qini(candidato) − Qini(baseline), por split:')
print(deltas_vs_baseline.round(4))

Neither leader showed a consistent advantage over the response-targeting baseline.

| Candidate | Mean | Median | Win rate |
|---|---|---|---|
| X+Tree(depth=4) | 0.0250 | 0.0259 | 40.0% |
| S+LightGBM(vanilla) | 0.0232 | 0.0228 | 40.0% |
| Response-targeting baseline | 0.0209 | 0.0172 | 20.0% |

Paired differences against the baseline, per split:

| Candidate | Mean Δ | Median Δ | P(Δ>0) |
|---|---|---|---|
| X+Tree(depth=4) | +0.0041 | +0.0007 | 53% |
| S+LightGBM(vanilla) | +0.0024 | +0.0013 | 60% |

Although X+Tree and S+LightGBM win 40% of repetitions each versus 20% for the baseline in raw win counts, paired differences show X+Tree beating the baseline in only 53% of splits (median Δ 0.0007) and S+LightGBM in 60% (median Δ 0.0013). As a secondary diagnostic on partially overlapping samples, paired Wilcoxon and t-tests yield p = 0.56 and 0.42 for X+Tree and p = 0.68 and 0.54 for S+LightGBM.

The central question of the subsection receives a negative answer: despite a slightly positive mean, the gain is small relative to sample variability and the baseline wins nearly half of the splits. The result is reported without further hyperparameter tuning (Absolute Rule #6).

<a id="s4-9-4"></a>

### 4.9.4 Causes of Qini AUC variation across samples

Across the three comparisons above, the standard deviation of Qini AUC under resampling (0.010 to 0.017) is large relative to the means (0.02 to 0.03) and between-candidate differences (often below 0.01). Three plausible explanations apply without formal isolation in this stage. First, each repetition trains on 75% of `train_df` (≈28,800 rows) and evaluates on 25% (≈9,600 rows), smaller than the fixed split (38,400 and 12,800 rows), increasing sensitivity of Qini AUC to how `visit` events fall across treated and control arms. Second, `visit` combines low-to-moderate prevalence with a small incremental effect of a few percentage points, making fine individual ranking metrics noisier than aggregate means such as ATE. Third, the models are refitted on every repetition, adding model-estimation variance (particularly in the two-stage X-learner) to evaluation variance.

<a id="s4-10"></a>

## 4.10 Linear S-learner with explicit T×X interactions (interpretable baseline)

In 4.6, `S+ElasticNet` yielded Qini AUC = 0.0000 because `causalml`'s S-learner appends treatment without covariate interactions, yielding `CATE = μ(x,1) − μ(x,0) = β_treatment`, a constant. This subsection tests, in a purely explanatory role without competing for 4.9 leadership, whether adding explicit `T×X_j` interactions recovers part of the signal captured by the shallow tree in an interpretable linear model.

Methodological issue identified in `causalml` source code: `BaseSRegressor.predict` runs two calls (`hstack([zeros, X])` and `hstack([ones, X])`) varying only the treatment column. Precomputed `T×X_j` columns inside `X` would remain identical in both calls and cancel out in subtraction. Accordingly, this S-learner was implemented directly in `src/learners.py` (`fit_s_learner_linear_interaction` / `predict_s_learner_linear_interaction_uplift`), fitting a single `ElasticNet` on `[X, T, T×X_1, ..., T×X_p]` and computing `CATE(x) = coef_T + Σ(coef_{T×X_j}·X_j)` in closed form.

Standardization caveat: features from `encode_meta_learner_features` are unstandardized, leaving `recency` (0–12) and `history` (0 to ~3,400, mean ~243) on much larger scales than binary indicators (0/1). Under L1/L2 penalization (`ElasticNet`), larger-scale features require smaller coefficient magnitudes for equal explanatory power, biasing feature survival against small-scale binaries. Original preprocessing and default hyperparameters (`alpha=1.0, l1_ratio=0.5`) are preserved to isolate only the addition of `T×X_j` terms (Absolute Rule #6), comparing `S+ElasticNet` with and without `T×X` and using `S+DecisionTree(depth=4)` only as a nonlinear reference.

In [ ]:
from sklearn.linear_model import ElasticNet
from sklearn.tree import DecisionTreeRegressor

from src.config import BIN_VARS, CAT_VARS, CONT_VARS
from src.evaluation import evaluate_multiple_rankings
from src.learners import (
    encode_meta_learner_features, fit_s_learner_linear_interaction, fit_single_meta_learner,
    predict_s_learner_linear_interaction_uplift, predict_single_meta_learner,
)

X_train_s410 = encode_meta_learner_features(train_df, meta_encoder)
X_val_s410 = encode_meta_learner_features(val_df, meta_encoder)
treatment_train_s410 = train_df[POOLED_TREATMENT_COL].to_numpy()
y_train_s410 = train_df[PRIMARY_OUTCOME].to_numpy(dtype=float)

s410_scores = {}

model_no_interaction = fit_single_meta_learner(
    'S', X_train_s410, treatment_train_s410, y_train_s410,
    base_learner_factory=lambda: ElasticNet(random_state=SEED),
)
s410_scores['S+ElasticNet (sem T×X)'] = predict_single_meta_learner('S', model_no_interaction, X_val_s410)

model_interaction = fit_s_learner_linear_interaction(
    X_train_s410, treatment_train_s410, y_train_s410, alpha=1.0, l1_ratio=0.5,
)
cate_interaction = predict_s_learner_linear_interaction_uplift(model_interaction, X_val_s410)
s410_scores['S+ElasticNet (com T×X)'] = cate_interaction

model_tree = fit_single_meta_learner(
    'S', X_train_s410, treatment_train_s410, y_train_s410,
    base_learner_factory=lambda: DecisionTreeRegressor(max_depth=4, random_state=SEED),
)
s410_scores['S+DecisionTree(depth=4)'] = predict_single_meta_learner('S', model_tree, X_val_s410)

s410_table = evaluate_multiple_rankings(val_df[PRIMARY_OUTCOME].values, s410_scores, val_df[POOLED_TREATMENT_COL].values)
print(s410_table.round(4))

n_features = X_train_s410.shape[1]
coef_t = model_interaction.coef_[n_features]
coef_interaction_vals = model_interaction.coef_[n_features + 1:]
cat_feature_names = list(meta_encoder.get_feature_names_out(CAT_VARS))
feature_names = CONT_VARS + BIN_VARS + cat_feature_names
nonzero_idx = coef_interaction_vals.nonzero()[0]

labels = lang({
    'header': 'Coeficientes do S+ElasticNet com T×X',
    'nonzero_header': 'Interações não nulas do ElasticNet',
})
print(f"\n{labels['header']}:")
print(f"  coef_T (efeito base) = {coef_t:.4f}")
print(f"  {labels['nonzero_header']}: {len(nonzero_idx)} de {len(coef_interaction_vals)}")
for i in nonzero_idx:
    print(f"    T×{feature_names[i]}: coef={coef_interaction_vals[i]:.6f}")

A non-zero signal was recovered, though the observed pattern is consistent with a scale artifact rather than an identified interaction:

| Configuration | Qini AUC | Uplift AUC | Uplift@30% |
|---|---|---|---|
| S+ElasticNet (without T×X) | 0.0000 | 0.0000 | 0.0626 |
| S+ElasticNet (with T×X) | 0.0229 | 0.0134 | 0.0808 |
| S+DecisionTree(depth=4) | 0.0515 | 0.0313 | 0.0894 |

Adding interactions lifts Qini AUC from 0.0000 to 0.0229 (about 45% of the shallow tree under the same S-learner format). However, `ElasticNet` (`alpha=1.0`, unstandardized) zeros 17 of the 18 interaction terms and shrinks `coef_T` to 0.0000, retaining only `T×history` with coefficient 0.000118. Because `history` has by far the largest raw scale (mean ≈243, std ≈256, max ≈3,346 versus 0/1 binaries and 0–12 `recency`), this confirms the prior L1-penalty caveat: without standardization, causal relevance cannot be separated from raw scale bias.

The result is reported without altering preprocessing in this round (Absolute Rule #6), recording standardization as future work. Linear interactions can recover part of the signal, but the unstandardized design cannot identify which covariate drives the interaction and does not alter the 4.9 direction.

<a id="s4-summary"></a>

### Synthesis of Section 4

Fixed-holdout results (single train-to-validation split, 4.1–4.8):

| Item | Result |
|---|---|
| Baseline (propensity) | Qini AUC 0.0395 |
| Best among the 4 original meta-learners (default LightGBM, 4.1–4.3) | S-learner, 0.0415 (minimal margin over baseline); X-learner (0.0243) below baseline, structural-advantage hypothesis not confirmed |
| Heterogeneity diagnostics (4.4) | Real CATE dispersion above permutation noise floor for S-learner, and Δ_GATES with 95% CI excluding zero for S and X; both exploratory diagnostics, neither is a formal test against homogeneous non-zero ATE |
| Derived features + regularization in X/R (4.5) | Regularization helps X (0.0243→0.0347); derived features help R (0.0196→0.0352); no configuration beats original S-learner; combining both axes worsens both |
| Sensitivity to base algorithm (4.6) | Single shallow tree (`max_depth=4`) beats LightGBM across all 4 meta-learners; X+tree reaches Qini AUC 0.0627 (+58.7% or 1.59× over baseline); unrestricted Random Forest gives negative Qini for S and T |
| Known propensity (2/3) vs. estimated in X/R (4.7) | Mostly null result: little effect on X-learner and slight lift on R-learner; does not explain the gap against S-learner with tree |
| Isolating bagging: single tree vs. RF at same depth (4.8) | Poor RF performance in 4.6 stemmed from unrestricted depth; at `max_depth=4`, RF turns positive across all 4 learners, though bagging worsens X and T |
| Linear S-learner with T×X interactions (4.10) | Leaves exact zero (0.0229), yet only `T×history` survives penalization, a pattern consistent with unstandardized ElasticNet scale bias |

Repeated stratified holdout results (15 75%/25% resamples inside `train_df`, without touching `val_df` or sealed test, 4.9):

| Item | Result |
|---|---|
| `max_depth` stability (4.9.1) | Mean Qini AUC between 0.0250 and 0.0269 for `depth` in `{3, 4, 5}`, differences smaller than standard deviations; `max_depth=4` kept for continuity with 4.6–4.8 |
| Paired comparison of leader (4.9.2) | `X+Tree(depth=4)` has the highest mean (0.0250) and win rate (46.7%), beating `T+Tree` (93% of splits) and regularized variants (67%), while tied with `S+LightGBM(vanilla)` (53% of splits, mean Δ 0.0018; Wilcoxon p = 0.89, t-test p = 0.71) |
| Leaders vs. response-targeting baseline (4.9.3) | `X+Tree` beats baseline in 53% of splits (mean Δ 0.0041) and `S+LightGBM` in 60% (mean Δ 0.0024); neither shows a consistent advantage |
| Qini variability (4.9.4) | Standard deviation of 0.010–0.017 against means of 0.02–0.03, plausibly linked to smaller sample size, low `visit` prevalence/effect, and refitting variance |

The central question in 4.9.3 regarding a consistent advantage of the leaders over the response-targeting baseline on the same resamples is answered negatively: despite a slightly positive mean, the difference is small relative to sample variability and the baseline wins nearly half of the repetitions, reported without compensatory tuning (Absolute Rule #6).

Conclusions retained after stability analysis: the T-learner showed strong cross-protocol instability (dropping from 0.0615 on the fixed holdout to 0.0039 average under resampling in `T+Tree`), which is more informative than labeling it uniformly worst; base-learner complexity control had a relevant impact in X/R regularization and Random Forest depth capping, although the data do not separate a specific shallow depth in `{3, 4, 5}` and comparisons share the same dataset; structure beyond pure noise is present (4.4) without formal confirmation of heterogeneity against a non-zero homogeneous ATE; and `X+Tree(depth=4)` and `S+LightGBM(vanilla)` form the top tier without stable separation, both advancing to S5/S6.

Previous conclusions downgraded or rejected: treating `X+Tree(depth=4)` (0.0627) as a stable winner was downgraded given sample sensitivity under resampling (mean ~0.025); consistent superiority of meta-learners over baseline was rejected in 4.9.3; attributing Random Forest's 4.6 failure to bagging was corrected in 4.8 as unrestricted depth (still limited to the fixed holdout); reading 4.4 as confirmed heterogeneity was corrected by distinguishing H0: τ(x)=0 from H0: τ(x)=ATE; and the claim of total base-algorithm dominance was softened to a material validation impact without stronger deterministic claims.

Remaining limitations: repeated stratified holdout shows high Qini AUC variance without formal isolation of noise sources; the four learners in Δ_GATES (4.4) evaluate the same ~12,800 individuals in `val_df` and are not independent evidence; the linear S-learner with interactions (4.10) is confounded by lack of feature standardization; no ATE-preserving permutation test was implemented; and all steps remain inside `train_df` and `val_df`, deferring confirmatory inference to the single sealed-test opening in S6.

Direction for S5 and S6: Section 4 selects no single winner. `X+Tree(depth=4)` and `S+LightGBM(vanilla)` advance as S5 references alongside the response-targeting baseline, where direct estimators (Causal Forest and Uplift Trees) will be evaluated under the same development protocols before final freezing at the end of S5 and single sealed-test evaluation in S6. Section 4 is frozen from this point onward, with no further depth tests, Random Forest variants, features, propensity variants, or meta-learners added.

Next: Section 5, Causal Forest and Uplift Trees.

---

<a id="glossary"></a>

## Glossary (quick reference)

| Term | Definition |
|---|---|
| ATE | Average Treatment Effect: average treatment effect on the population |
| ITE | Individual Treatment Effect: individual effect (unobservable) |
| CATE | Conditional Average Treatment Effect: average treatment effect conditional on $X$, the target of uplift modeling |
| SMD | Standardized Mean Difference: metric of balance between groups |
| Uplift score | Estimate of $\tau(x)$ produced by a model |
| Persuadable | Customer with relevant positive uplift |
| Sure Thing | Customer who converts with or without treatment (uplift ≈ 0) |
| Lost Cause | Customer who does not convert in any scenario (uplift ≈ 0) |
| Sleeping Dog | Customer with negative uplift (treatment is detrimental) |
| Qini coefficient | Metric of uplift evaluation analogous to Gini, integrates the Qini curve |
| AUUC | Area Under the Uplift Curve: equivalent to AUC for uplift |

## References

Breiman, L. (2001). Random forests. *Machine Learning*, 45(1), 5–32. DOI: [10.1023/A:1010933404324](https://doi.org/10.1023/A:1010933404324)

Breiman, L., Friedman, J. H., Olshen, R. A., & Stone, C. J. (1984). *Classification and Regression Trees*. Wadsworth. Reprinted by Routledge (2017). https://doi.org/10.1201/9781315139470

Chernozhukov, V., Demirer, M., Duflo, E., & Fernández-Val, I. (2018b). *Generic Machine Learning Inference on Heterogeneous Treatment Effects in Randomized Experiments* (NBER Working Paper 24678). National Bureau of Economic Research. DOI: [10.3386/w24678](https://doi.org/10.3386/w24678)

Efron, B., & Tibshirani, R. J. (1993). *An Introduction to the Bootstrap*. Chapman & Hall. DOI: [10.1007/978-1-4899-4541-9](https://doi.org/10.1007/978-1-4899-4541-9)

Gutierrez, P., & Gérardy, J.-Y. (2017). Causal inference and uplift modelling: A review of the literature. *Proceedings of Machine Learning Research*, 67, 1–13. [https://proceedings.mlr.press/v67/gutierrez17a.html](https://proceedings.mlr.press/v67/gutierrez17a.html)

Hillstrom, K. (2008). *The MineThatData E-Mail Analytics and Data Mining Challenge*. MineThatData blog. [https://blog.minethatdata.com/2008/03/minethatdata-e-mail-analytics-and-data.html](https://blog.minethatdata.com/2008/03/minethatdata-e-mail-analytics-and-data.html)

Imbens, G. W., & Rubin, D. B. (2015). *Causal Inference for Statistics, Social, and Biomedical Sciences: An Introduction*. Cambridge University Press. DOI: [10.1017/CBO9781139025751](https://doi.org/10.1017/CBO9781139025751)

Ke, G., Meng, Q., Finley, T., Wang, T., Chen, W., Ma, W., Ye, Q., & Liu, T.-Y. (2017). LightGBM: A highly efficient gradient boosting decision tree. In *Advances in Neural Information Processing Systems 30* (pp. 3146–3154). [https://papers.nips.cc/paper/6907-lightgbm-a-highly-efficient-gradient-boosting-decision-tree](https://papers.nips.cc/paper/6907-lightgbm-a-highly-efficient-gradient-boosting-decision-tree)

Künzel, S. R., Sekhon, J. S., Bickel, P. J., & Yu, B. (2019). Metalearners for estimating heterogeneous treatment effects using machine learning. *Proceedings of the National Academy of Sciences*, 116(10), 4156–4165. DOI: [10.1073/pnas.1804597116](https://doi.org/10.1073/pnas.1804597116)

Nie, X., & Wager, S. (2021). Quasi-oracle estimation of heterogeneous treatment effects. *Biometrika*, 108(2), 299–319. DOI: [10.1093/biomet/asaa076](https://doi.org/10.1093/biomet/asaa076)

Radcliffe, N. J., & Surry, P. D. (2011). *Real-World Uplift Modelling with Significance-Based Uplift Trees* (White Paper TR-2011-1). Stochastic Solutions. [https://stochasticsolutions.com/pdf/sig-based-up-trees.pdf](https://stochasticsolutions.com/pdf/sig-based-up-trees.pdf)

Rosenbaum, P. R., & Rubin, D. B. (1983). The central role of the propensity score in observational studies for causal effects. *Biometrika*, 70(1), 41–55. DOI: [10.1093/biomet/70.1.41](https://doi.org/10.1093/biomet/70.1.41)

Zou, H., & Hastie, T. (2005). Regularization and variable selection via the elastic net. *Journal of the Royal Statistical Society: Series B*, 67(2), 301–320. DOI: [10.1111/j.1467-9868.2005.00503.x](https://doi.org/10.1111/j.1467-9868.2005.00503.x)